# MIG Calculator for Jobstats (v0.0.0)
### Last Updated: March 30, 2026
### Questions? halverson@princeton.edu

This Jupyter notebook can be used to calculate whether or not a cluster with NVIDIA GPUs will benefit from MIG. If it will benefit then this notebook calculates how many GPUs to convert. The person running this notebook must enter the cluster name, partitions, and time window. They must also enter thresholds for the GPU efficiency, GPU memory usage, and number of CPU-cores toward the bottom of the notebook under "Analysis of the data".

## Setup

The pandas Python package is required to use this notebook. The next line downloads a necessary Python module:

In [ ]:
!wget https://raw.githubusercontent.com/PrincetonUniversity/job_defense_shield/refs/heads/main/src/job_defense_shield/efficiency.py

In [ ]:
import os
import re
import subprocess
from datetime import datetime
import pandas as pd
from efficiency import get_stats_dict
from efficiency import cpu_memory_usage
from efficiency import gpu_memory_usage_eff_tuples

In [ ]:
# convert slurm timestamps to seconds
os.environ["SLURM_TIME_FORMAT"] = "%s"

In [ ]:
def get_data_from_sacct(clusters: str,
                        partition: str,
                        start_date: str,
                        end_date: str,
                        fields: str) -> pd.DataFrame:
    """Return a dataframe of the sacct output."""
    partition = f"-r {partition}" if partition else partition
    cmd = f"sacct -M {clusters} {partition} -a -X -P -n -S {start_date} -E {end_date} -o {fields}"
    output = subprocess.run(cmd,
                            stdout=subprocess.PIPE,
                            shell=True,
                            timeout=100,
                            text=True,
                            check=True)
    rows = [row.split("|") for row in output.stdout.strip().split()]
    df = pd.DataFrame(rows)
    df.columns = fields.split(",")
    return df

In [ ]:
def days_between(date_str1, date_str2):
    """
    Calculates the absolute number of days between two date strings 
    in 'YYYY-MM-DD' format.
    """
    date_format = "%Y-%m-%d"
    d1 = datetime.strptime(date_str1, date_format)
    d2 = datetime.strptime(date_str2, date_format)
    delta = abs(d2 - d1)
    return delta.days

## The job data is obtained and cleaned

Enter the cluster name, a comma-separated list of the partitions of interest, and the time window:

In [ ]:
clusters = "della"
partition = "gpu,llm"  # use an empty string for all partitions
start_date = "2026-03-23"
end_date = "2026-03-30"  # do not use "now"

You can skip to the last section of the notebook.

In [ ]:
fields = "jobid,cluster,user,alloctres,elapsedraw,admincomment,ncpus"
df = get_data_from_sacct(clusters, partition, start_date, end_date, fields)
days = days_between(start_date, end_date)
hours_per_day = 24

In [ ]:
df = df[pd.notna(df.elapsedraw)]
df = df[df.elapsedraw.str.isnumeric()]
df.elapsedraw = df.elapsedraw.astype("int64")
df = df[df.elapsedraw > 0]

In [ ]:
df = df[pd.notna(df.ncpus)]
df = df[df.ncpus.str.isnumeric()]
df.ncpus = df.ncpus.astype("int64")
df = df[df.ncpus > 0]

## New fields are added

In [ ]:
def gpus_per_job(tres: str) -> int:
    """Return the number of allocated GPUs."""
    gpus = re.findall(r"gres/gpu=\d+", tres)
    return int(gpus[0].replace("gres/gpu=", "")) if gpus else 0

In [ ]:
df["gpus"] = df.alloctres.apply(gpus_per_job)
df["gpu-seconds"] = df.apply(lambda row: row["elapsedraw"] * row["gpus"], axis='columns')
df["gpu-hours"] = df["gpu-seconds"] / 3600
df["admincomment"] = df["admincomment"].apply(get_stats_dict)

In [ ]:
df["gpu-tuple"] = df.apply(lambda row: gpu_memory_usage_eff_tuples(row["admincomment"],
                                                                   row["jobid"],
                                                                   row["cluster"],
                                                                   verbose=False),
                                                                   axis="columns")

In [ ]:
df["error_code"] = df["gpu-tuple"].apply(lambda x: x[1])
df = df[df["error_code"] == 0]
df["GPU-Mem-Used"] = df["gpu-tuple"].apply(lambda tpl: tpl[0][0][0])
df["GPU-Util"]     = df["gpu-tuple"].apply(lambda tpl: tpl[0][0][2])
df.drop(columns=["error_code"], inplace=True)

In [ ]:
df["memory-tuple"] = df.apply(lambda row: cpu_memory_usage(row["admincomment"],
                                                           row["jobid"],
                                                           row["cluster"],
                                                           verbose=False),
                                                           axis="columns")
cols = ["CPU-Mem-Used", "mem-alloc", "error_code"]
df[cols] = pd.DataFrame(df["memory-tuple"].tolist(), index=df.index)
df = df[df["error_code"] == 0]

In [ ]:
df["cores_per_gpu"] = df.ncpus / df.gpus
df["CPU-Mem-Used-per-GPU"] = df["CPU-Mem-Used"] / df.gpus

In [ ]:
def max_gpu_mem(tpl):
    items, error_code = tpl
    return max([item[0] for item in items])

In [ ]:
def max_gpu_util(tpl):
    items, error_code = tpl
    return max([item[2] for item in items])

In [ ]:
df["max_gpu_mem"]  = df["gpu-tuple"].apply(max_gpu_mem)
df["max_gpu_util"] = df["gpu-tuple"].apply(max_gpu_util)

## About your data

Below are the first two rows of the data frame as a sanity check:

In [ ]:
df.head(2).T

The table below contains simple statistics about your data:

In [ ]:
df.describe().round(1)

## Analysis of the data

Enter the number of GPUs on your cluster (corresponding to the cluster and partitions entered above):

In [ ]:
num_gpus = 128

The calculation below find the percentag of GPU-hours associated with jobs with 10 allocated CPU-cores per gpu or less, less than 60 GB of used CPU memory, a GPU utilization of less than 50%, and less than 40 GB of used GPU memory:

In [ ]:
s = df[(df["cores_per_gpu"] <= 10) & (df["CPU-Mem-Used-per-GPU"] < 60) & (df["max_gpu_util"] < 50) & (df["max_gpu_mem"] < 40)]["gpu-hours"].sum()
pct = f"Percentage of GPU-hours that could run on 40 GB MIG instance = {round(100 * s / (days * hours_per_day * num_gpus))}%"
print(pct)

Below is a different calculation that only considers single-GPU jobs. It calculates the percentage of GPU-hours associated with jobs with 16 allocated CPU-cores per gpu or less, less than 60 GB of used CPU memory, a GPU utilization of less than 70%, and less than 40 GB of used GPU memory:

In [ ]:
s = df[(df.gpus == 1) & (df["cores_per_gpu"] <= 16) & (df["CPU-Mem-Used-per-GPU"] < 60) & (df["max_gpu_util"] < 70) & (df["max_gpu_mem"] < 40)]["gpu-hours"].sum()
pct = f"Percentage of GPU-hours that could run on 40 GB MIG instance = {round(100 * s / (days * hours_per_day * num_gpus))}%"
print(pct)

Knowing the percentage of GPU-hours, one can calculation the number of full GPUs to convert to MIG.